In [ ]:
%%html

<!-- UI_RUNNER: Data Structures for Graph Heuristics Zones -->

<script>

class AlgoZone {
  constructor(code, title, subtitle, description) {
    this.code = code;
    this.title = title;
    this.subtitle = subtitle;
    this.description = description;
  }
}

window.algoZones = {
  greedy: new AlgoZone(
    'greedy',
    'Greedy BFS',
    'h(n) only — fast, not optimal',
    'Uses only the heuristic to greedily pick the next node. Fast but may miss the optimal path.'
  ),
  dijkstra: new AlgoZone(
    'dijkstra',
    "Dijkstra's",
    'g(n) only — optimal, exhaustive',
    'Guarantees shortest path by expanding cheapest-cost nodes first. No heuristic used.'
  ),
  astar: new AlgoZone(
    'astar',
    'A* Search',
    'f(n) = g(n) + h(n) — best of both',
    'Combines actual cost and heuristic estimate. Optimal and efficient when h(n) is admissible.'
  )
};

window.AlgoZone = AlgoZone;

</script>

In [ ]:
%%html

<!-- UI_RUNNER: Graph Heuristics Introduction -->

<div style="padding: 1.25rem 1.5rem; background: linear-gradient(135deg, #0f0f23 0%, #1a1a3a 100%); border: 1px solid #2a7ae4; border-radius: 8px; margin-bottom: 1.5rem; color: #ccc; font-family: var(--pref-font-family, inherit); line-height: 1.7;">
  <h1 style="font-size: 1.6rem; margin: 0 0 0.5rem 0; color: #7ab8f5;">Graph Heuristics</h1>
  <p style="font-size: 1rem; margin: 0 0 1rem 0; color: #aaa; font-style: italic;">Greedy BFS &bull; Dijkstra's &bull; A*</p>
  <p style="margin: 0 0 0.75rem 0; font-size: 0.95rem;">
    A <strong style="color:#eee;">graph heuristic</strong> is a strategy for deciding which node to explore next when searching through a graph.
    The three algorithms in this lesson each make that decision differently — trading off speed, memory, and optimality.
  </p>
  <div style="display:flex; gap:1rem; flex-wrap:wrap; margin-top:0.75rem;">
    <div style="flex:1; min-width:160px; background:#1a1a2e; border:1px solid #333; border-radius:6px; padding:0.75rem;">
      <div style="font-weight:600; color:#f5c842; margin-bottom:0.25rem;">Greedy BFS</div>
      <div style="font-size:0.82rem; color:#aaa;">Uses only <code style="background:#2a2a2a;padding:1px 4px;border-radius:3px;color:#e6c97a;">h(n)</code> — the estimated cost to goal. Fast but not guaranteed optimal.</div>
    </div>
    <div style="flex:1; min-width:160px; background:#1a1a2e; border:1px solid #333; border-radius:6px; padding:0.75rem;">
      <div style="font-weight:600; color:#42c8f5; margin-bottom:0.25rem;">Dijkstra's</div>
      <div style="font-size:0.82rem; color:#aaa;">Uses only <code style="background:#2a2a2a;padding:1px 4px;border-radius:3px;color:#e6c97a;">g(n)</code> — the actual cost from start. Optimal but explores in all directions.</div>
    </div>
    <div style="flex:1; min-width:160px; background:#1a1a2e; border:1px solid #333; border-radius:6px; padding:0.75rem;">
      <div style="font-weight:600; color:#42f587; margin-bottom:0.25rem;">A*</div>
      <div style="font-size:0.82rem; color:#aaa;">Uses <code style="background:#2a2a2a;padding:1px 4px;border-radius:3px;color:#e6c97a;">f(n) = g(n) + h(n)</code> — combines both. Optimal <em>and</em> efficient. Used in the Maze Runner.</div>
    </div>
  </div>
  <p style="margin: 1rem 0 0 0; font-size:0.85rem; color:#888;">
    Work through the sidebar from left to right. Pass the quiz for each algorithm to unlock the next.
  </p>
</div>

In [ ]:
%%html

<!-- UI_RUNNER: Helper classes for gamified progress tracking -->

<script>

class ZoneStateManager {
  constructor(storageKey) {
    this.storageKey = storageKey || 'graphHeuristicsV2';
    this.unlocked = {};
    this.selectedZone = null;
    this.load();
    // Greedy always starts unlocked
    if (!this.unlocked['greedy']) {
      this.unlocked['greedy'] = true;
      this.save();
    }
  }

  load() {
    var saved = localStorage.getItem(this.storageKey);
    if (saved) {
      try {
        var data = JSON.parse(saved);
        this.unlocked = data.unlocked || {};
        this.selectedZone = data.selectedZone || null;
      } catch (e) {}
    }
  }

  save() {
    localStorage.setItem(this.storageKey, JSON.stringify({
      unlocked: this.unlocked,
      selectedZone: this.selectedZone
    }));
  }

  clear() {
    localStorage.removeItem(this.storageKey);
    this.unlocked = { greedy: true };
    this.selectedZone = null;
  }

  selectZone(zoneCode) {
    this.selectedZone = zoneCode;
    this.save();
  }

  isUnlocked(zoneCode) {
    return !!this.unlocked[zoneCode];
  }

  setUnlocked(zoneCode, val) {
    this.unlocked[zoneCode] = val;
    this.save();
  }
}

window.ZoneStateManager = ZoneStateManager;

</script>

In [ ]:
%%html

<!-- UI_RUNNER: Algorithm Progress Tracker | panel: heuristics_journey, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<div id="zone-unlock-panel">
  <h3>Algorithm Journey</h3>
  <p class="zone-subtitle">Pass each quiz to unlock the next algorithm. <button id="clear-progress-btn" style="margin-left:8px;padding:3px 7px;font-size:0.8em;">Reset</button></p>
  <div id="zone-buttons-container"></div>
</div>

<script>

class ZoneButtonRenderer {
  constructor(stateManager, zones) {
    this.stateManager = stateManager;
    this.zones = zones;
    this.renderButtons();
  }

  renderButtons() {
    var container = document.getElementById('zone-buttons-container');
    if (!container) return;
    container.innerHTML = '';

    var zoneKeys = Object.keys(this.zones);
    zoneKeys.forEach(function(zoneCode) {
      var zone = this.zones[zoneCode];
      var button = document.createElement('button');
      button.className = 'zone-btn locked';
      button.setAttribute('data-zone', zoneCode);
      button.disabled = true;
      button.innerHTML =
        '<div class="zone-btn-title">' + zone.title + '</div>' +
        '<div class="zone-btn-subtitle">' + zone.subtitle + '</div>' +
        '<div class="zone-btn-lock">🔒 Complete previous quiz</div>';
      container.appendChild(button);
    }.bind(this));
  }

  updateAllButtons() {
    var zoneKeys = Object.keys(this.zones);
    zoneKeys.forEach(function(zoneCode) {
      this.updateButton(zoneCode);
    }.bind(this));
    this.updateZoneSectionVisibility();
  }

  updateButton(zoneCode) {
    var zone = this.zones[zoneCode];
    var unlocked = this.stateManager.isUnlocked(zoneCode);
    var btn = document.querySelector('.zone-btn[data-zone="' + zoneCode + '"]');
    if (!btn) return;

    if (unlocked) {
      btn.classList.remove('locked');
      btn.classList.add('unlocked');
      btn.disabled = false;
      var lockDiv = btn.querySelector('.zone-btn-lock');
      if (lockDiv) lockDiv.textContent = '✅ Unlocked';
    } else {
      btn.classList.remove('unlocked');
      btn.classList.add('locked');
      btn.disabled = true;
      var lockDiv2 = btn.querySelector('.zone-btn-lock');
      if (lockDiv2) lockDiv2.textContent = '🔒 Complete previous quiz';
    }

    if (this.stateManager.selectedZone === zoneCode) {
      btn.classList.add('active');
    } else {
      btn.classList.remove('active');
    }
  }

  updateZoneSectionVisibility() {
    var zoneKeys = Object.keys(this.zones);
    var sm = this.stateManager;
    zoneKeys.forEach(function(zoneCode) {
      var section = document.getElementById('zone-section-' + zoneCode);
      var isActive = sm.selectedZone === zoneCode && sm.isUnlocked(zoneCode);
      if (section) {
        section.style.display = isActive ? 'block' : 'none';
      }
    });
  }
}

var stateManager = new window.ZoneStateManager('graphHeuristicsV2');
var buttonRenderer = new ZoneButtonRenderer(stateManager, window.algoZones);
window.heuristicsButtonRenderer = buttonRenderer;
window.heuristicsStateManager = stateManager;

document.addEventListener('click', function(e) {
  if (e.target.id === 'clear-progress-btn') {
    stateManager.clear();
    buttonRenderer.updateAllButtons();
    return;
  }
  var btn = e.target.closest('.zone-btn');
  if (!btn || btn.disabled) return;
  var zone = btn.getAttribute('data-zone');
  stateManager.selectZone(zone);
  buttonRenderer.updateAllButtons();
});

window.addEventListener('quiz-complete', function(e) {
  var detail = e.detail || {};
  if (detail.nextZone) {
    stateManager.setUnlocked(detail.nextZone, true);
  }
  buttonRenderer.updateAllButtons();
});

buttonRenderer.updateAllButtons();

</script>

In [ ]:
%%js

// GAME_RUNNER: Maze Runner | hide_edit: true, panel: heuristics_journey, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 400px

import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelGraphHeuristics from '/assets/js/GameEnginev1.1/GameLevelGraphHeuristics.js';

export const gameLevelClasses = [GameLevelGraphHeuristics];
export { GameControl };


* Select an algorithm in the sidebar to begin. Complete each quiz to unlock the next.
* Use the **Maze Runner** on the right to see these algorithms in action!

---

{% comment %} Greedy Zone - Start {% endcomment %}

<div id="zone-section-greedy" class="zone-section" style="display: none;" markdown="1">

In [ ]:
%%html

<!-- UI_RUNNER: Greedy BFS Lesson -->

<style>
.lesson { color: var(--pref-text-color, #ccc); line-height: 1.7; font-family: var(--pref-font-family, inherit); }
.lesson h2 { font-size: 1.4rem; margin: 0 0 1rem 0; color: var(--pref-accent-color, #7ab8f5); border-bottom: 2px solid var(--pref-accent-color, #7ab8f5); padding-bottom: 0.4rem; }
.lesson h3 { font-size: 1.05rem; margin: 1.2rem 0 0.5rem 0; color: var(--pref-text-color, #ddd); font-weight: 600; }
.lesson p { margin: 0.5rem 0; font-size: 0.95rem; }
.lesson ol, .lesson ul { padding-left: 1.4rem; margin: 0.5rem 0; }
.lesson li { margin: 0.3rem 0; font-size: 0.95rem; }
.lesson table { width: 100%; border-collapse: collapse; margin: 0.75rem 0; font-size: 0.88rem; }
.lesson th { background: #2a2a3a; padding: 0.4rem 0.75rem; text-align: left; border: 1px solid #444; color: #adf; }
.lesson td { padding: 0.4rem 0.75rem; border: 1px solid #333; }
.lesson code { background: #2a2a2a; padding: 0.1rem 0.4rem; border-radius: 3px; font-size: 0.85em; color: #e6c97a; font-family: monospace; }
.lesson blockquote { border-left: 3px solid var(--pref-accent-color, #7ab8f5); padding: 0.5rem 1rem; margin: 0.75rem 0; background: #1a1a2a; border-radius: 0 4px 4px 0; }
.lesson .callout { background: #1a2a1a; border: 1px solid #3a6a3a; border-radius: 6px; padding: 0.75rem 1rem; margin: 0.75rem 0; font-size: 0.9rem; }
.lesson strong { color: var(--pref-text-color, #eee); }
</style>

<div class="lesson">
  <h2>Greedy Best-First Search</h2>
  <p>Greedy Best-First Search expands the node that <strong>appears closest to the goal</strong> using only a heuristic function <code>h(n)</code>. It never looks back at how far it has already traveled — only at how far it thinks it needs to go.</p>

  <h3>Core idea</h3>
  <p>At every step, look at all available nodes and pick the one with the <strong>smallest h(n)</strong> (estimated distance to goal). This is "greedy" because it commits immediately to what looks best right now.</p>

  <h3>Algorithm steps</h3>
  <ol>
    <li>Add the start node to the open list (priority queue sorted by <code>h(n)</code>)</li>
    <li>Pop the node with the lowest <code>h(n)</code></li>
    <li>If it's the goal — done. Otherwise mark it visited</li>
    <li>Add all unvisited neighbors to the open list</li>
    <li>Repeat from step 2</li>
  </ol>

  <h3>Key properties</h3>
  <table>
    <tr><th>Property</th><th>Greedy BFS</th></tr>
    <tr><td><strong>Uses</strong></td><td><code>h(n)</code> only — estimated cost to goal</td></tr>
    <tr><td><strong>Complete?</strong></td><td>No — can loop in graphs with cycles</td></tr>
    <tr><td><strong>Optimal?</strong></td><td>No — ignores actual edge costs</td></tr>
    <tr><td><strong>Time</strong></td><td>O(b<sup>m</sup>) worst case</td></tr>
    <tr><td><strong>Space</strong></td><td>O(b<sup>m</sup>) worst case</td></tr>
  </table>

  <h3>Why it can fail</h3>
  <p>The heuristic is just an estimate. Greedy can take a path that looks short but is actually expensive. In the code below, the graph has two paths to the goal:</p>
  <ul>
    <li><strong>A → C → G</strong>: Greedy picks this (C has lower <code>h</code>) — total cost <strong>9</strong></li>
    <li><strong>A → B → G</strong>: Optimal path — total cost <strong>5</strong></li>
  </ul>
  <p>Greedy chose wrong because it only saw the heuristic, not the edge weights.</p>

  <div class="callout">
    <strong>AP Exam tip:</strong> On the AP CSA exam, Greedy algorithms appear in the context of locally optimal choices that don't guarantee a globally optimal solution. Know that Greedy ≠ optimal.
  </div>
</div>

In [ ]:
// CODE_RUNNER: Greedy Best-First Search

import java.util.*;

public class Main {

    static class Node {
        String name;
        int heuristic; // estimated distance to goal
        List<int[]> neighbors = new ArrayList<>(); // [neighborIndex, edgeCost]

        Node(String name, int heuristic) {
            this.name = name;
            this.heuristic = heuristic;
        }
    }

    public static void main(String[] args) {
        // Build graph: A(h=10) -> B(h=5), C(h=3); B -> G(h=0); C -> G(h=0)
        Node a = new Node("A", 10);
        Node b = new Node("B", 5);
        Node c = new Node("C", 3);
        Node g = new Node("G", 0);
        Node[] nodes = {a, b, c, g};

        a.neighbors.add(new int[]{1, 2}); // A->B cost 2
        a.neighbors.add(new int[]{2, 8}); // A->C cost 8
        b.neighbors.add(new int[]{3, 3}); // B->G cost 3
        c.neighbors.add(new int[]{3, 1}); // C->G cost 1

        // Greedy BFS: always pick the neighbor with lowest h(n)
        Set<String> visited = new HashSet<>();
        PriorityQueue<Integer> open = new PriorityQueue<>(
            Comparator.comparingInt(i -> nodes[i].heuristic)
        );

        open.add(0); // start at A
        Map<Integer, Integer> parent = new HashMap<>();
        parent.put(0, -1);

        System.out.println("=== Greedy BFS Traversal ===");
        int goalIdx = -1;
        while (!open.isEmpty()) {
            int curr = open.poll();
            if (visited.contains(nodes[curr].name)) continue;
            visited.add(nodes[curr].name);
            System.out.println("Visiting: " + nodes[curr].name + " (h=" + nodes[curr].heuristic + ")");

            if (nodes[curr].heuristic == 0) { goalIdx = curr; break; }

            for (int[] edge : nodes[curr].neighbors) {
                if (!visited.contains(nodes[edge[0]].name)) {
                    open.add(edge[0]);
                    if (!parent.containsKey(edge[0])) parent.put(edge[0], curr);
                }
            }
        }

        // Reconstruct path
        System.out.print("\nPath found: ");
        List<String> path = new ArrayList<>();
        int p = goalIdx;
        while (p != -1) { path.add(nodes[p].name); p = parent.get(p); }
        Collections.reverse(path);
        System.out.println(String.join(" -> ", path));

        System.out.println("\nNotice: Greedy chose C (h=3) over B (h=5) — lower heuristic!");
        System.out.println("Greedy path cost: 8 + 1 = 9");
        System.out.println("Optimal path cost: 2 + 3 = 5  (A->B->G)");
        System.out.println("Greedy is NOT optimal — it ignored actual edge costs.");
    }
}
Main.main(null);

In [ ]:
%%html

<!-- UI_RUNNER: Greedy BFS Knowledge Check -->

<style>
.quiz-container { margin: 1.5rem 0; }
.question { margin-bottom: 1rem; padding: 1rem; background: #1a1a1a; border: 1px solid #333; border-radius: 6px; border-left: 4px solid #2a7ae4; }
.question-title { font-weight: 600; font-size: 0.95rem; color: #fff; margin-bottom: 0.75rem; }
.question label { display: block; padding: 0.3rem 0; color: #aaa; cursor: pointer; font-size: 0.85rem; }
.question label:hover { color: #fff; }
.quiz-btn { padding: 0.6rem 1.4rem; background: #2a7ae4; color: #fff; border: none; border-radius: 4px; cursor: pointer; font-size: 0.95rem; margin-top: 0.5rem; }
.quiz-result { margin-top: 1rem; padding: 0.75rem 1rem; border-radius: 4px; font-weight: 600; font-size: 0.9rem; }
.unlock-notice { display: none; margin-top: 1rem; padding: 0.75rem 1rem; background: #1a2a3a; border: 1px solid #2a7ae4; border-radius: 4px; color: #7ab8f5; font-size: 0.9rem; }
.unlock-notice.show { display: block; }
</style>

<div class="quiz-container" id="greedy-quiz-container">
  <h3 style="margin-bottom:0.75rem;font-size:1.05rem;">Knowledge Check: Greedy BFS</h3>

  <div class="question">
    <div class="question-title">Q1. Greedy BFS selects the next node based on...</div>
    <label><input type="radio" name="qg1" value="a"> a) The smallest g(n) &mdash; actual cost from start</label>
    <label><input type="radio" name="qg1" value="b"> b) The smallest h(n) &mdash; estimated cost to goal</label>
    <label><input type="radio" name="qg1" value="c"> c) f(n) = g(n) + h(n)</label>
    <label><input type="radio" name="qg1" value="d"> d) Random neighbor selection</label>
  </div>

  <div class="question">
    <div class="question-title">Q2. Which statement about Greedy BFS is true?</div>
    <label><input type="radio" name="qg2" value="a"> a) It always finds the shortest path</label>
    <label><input type="radio" name="qg2" value="b"> b) It considers actual path costs</label>
    <label><input type="radio" name="qg2" value="c"> c) It may find a suboptimal path if the heuristic is misleading</label>
    <label><input type="radio" name="qg2" value="d"> d) It is guaranteed to be complete in all graphs</label>
  </div>

  <button class="quiz-btn" id="greedy-quiz-btn">Submit</button>
  <div id="greedy-quiz-result" class="quiz-result" style="display:none;"></div>
  <div id="greedy-quiz-unlock" class="unlock-notice">&#x2705; Dijkstra's Algorithm unlocked! Click it in the sidebar to continue.</div>
</div>

<script>
(function() {
  var btn = document.getElementById('greedy-quiz-btn');
  if (!btn) return;
  btn.addEventListener('click', function() {
    var container = document.getElementById('greedy-quiz-container');
    var q1 = container.querySelector('input[name="qg1"]:checked');
    var q2 = container.querySelector('input[name="qg2"]:checked');
    var result = document.getElementById('greedy-quiz-result');
    var unlock = document.getElementById('greedy-quiz-unlock');

    if (!q1 || !q2) {
      result.style.display = 'block';
      result.style.background = '#3a1a1a';
      result.style.border = '1px solid #f44336';
      result.style.color = '#f44336';
      result.textContent = 'Please answer all questions.';
      return;
    }

    var score = 0;
    if (q1.value === 'b') score++;
    if (q2.value === 'c') score++;

    result.style.display = 'block';
    if (score === 2) {
      result.style.background = '#1a3a1a';
      result.style.border = '1px solid #4CAF50';
      result.style.color = '#4CAF50';
      result.textContent = 'Perfect! 2/2 — You understand Greedy BFS.';
      unlock.style.display = 'block';
      window.dispatchEvent(new CustomEvent('quiz-complete', {
        detail: { zone: 'greedy', score: 2, nextZone: "dijkstra" }
      }));
    } else {
      result.style.background = '#3a1a1a';
      result.style.border = '1px solid #f44336';
      result.style.color = '#f44336';
      result.textContent = score + '/2 — Review the material above and try again.';
    }
  });
})();
</script>

{% comment %} Greedy Zone - End {% endcomment %}

</div><!-- closes #zone-section-greedy -->

{% comment %} Dijkstra Zone - Start {% endcomment %}

<div id="zone-section-dijkstra" class="zone-section" style="display: none;" markdown="1">

In [ ]:
%%html

<!-- UI_RUNNER: Dijkstra Lesson -->

<div class="lesson">
  <h2>Dijkstra's Algorithm</h2>
  <p>Dijkstra's algorithm finds the <strong>shortest path from a source node to every other node</strong> in a weighted graph. Unlike Greedy BFS, it tracks the actual cost traveled — not just an estimate.</p>

  <h3>Core idea</h3>
  <p>Maintain a running cost <code>g(n)</code> for each node — the cheapest known path from the start. Always expand the node with the <strong>smallest current g(n)</strong>. When a cheaper route to a neighbor is found, update it.</p>

  <h3>Algorithm steps</h3>
  <ol>
    <li>Set <code>g(start) = 0</code>, set all other nodes to <code>g = ∞</code></li>
    <li>Add start to a min-priority queue (sorted by <code>g</code>)</li>
    <li>Pop the node with the smallest <code>g(n)</code> — call it <em>current</em></li>
    <li>For each neighbor: compute <code>tentative = g(current) + edge_cost</code></li>
    <li>If <code>tentative &lt; g(neighbor)</code>, update and re-add to queue</li>
    <li>Mark current as settled. Repeat from step 3</li>
  </ol>

  <h3>Key properties</h3>
  <table>
    <tr><th>Property</th><th>Dijkstra's</th></tr>
    <tr><td><strong>Uses</strong></td><td><code>g(n)</code> only — actual cost from start</td></tr>
    <tr><td><strong>Complete?</strong></td><td>Yes</td></tr>
    <tr><td><strong>Optimal?</strong></td><td>Yes — for non-negative edge weights</td></tr>
    <tr><td><strong>Time</strong></td><td>O((V + E) log V) with a binary heap</td></tr>
    <tr><td><strong>Heuristic</strong></td><td>None — explores all directions equally</td></tr>
  </table>

  <h3>The tradeoff vs A*</h3>
  <p>Because Dijkstra has no heuristic, it expands outward in all directions like a ripple. This guarantees finding the shortest path to <strong>every</strong> node, but wastes time exploring nodes far from the goal when you only need one specific target.</p>

  <div class="callout">
    <strong>Real-world use:</strong> Network routing protocols (like OSPF) use Dijkstra's to find shortest paths across all routers. GPS systems prefer A* for single-destination navigation since it's faster.
  </div>
</div>

In [ ]:
// CODE_RUNNER: Dijkstra's Algorithm

import java.util.*;

public class Main {

    static class Edge {
        String to;
        int cost;
        Edge(String to, int cost) { this.to = to; this.cost = cost; }
    }

    public static void main(String[] args) {
        // Graph:  A --5-- B
        //         |       |
        //         1       2
        //         |       |
        //         C --2-- D
        Map<String, List<Edge>> graph = new HashMap<>();
        graph.put("A", Arrays.asList(new Edge("B", 5), new Edge("C", 1)));
        graph.put("B", Arrays.asList(new Edge("A", 5), new Edge("D", 2)));
        graph.put("C", Arrays.asList(new Edge("A", 1), new Edge("D", 2)));
        graph.put("D", Arrays.asList(new Edge("B", 2), new Edge("C", 2)));

        String start = "A";

        Map<String, Integer> dist = new HashMap<>();
        Map<String, String> prev = new HashMap<>();
        for (String node : graph.keySet()) dist.put(node, Integer.MAX_VALUE);
        dist.put(start, 0);

        PriorityQueue<String> pq = new PriorityQueue<>(Comparator.comparingInt(dist::get));
        pq.add(start);
        Set<String> settled = new HashSet<>();

        System.out.println("=== Dijkstra's Algorithm ===");
        while (!pq.isEmpty()) {
            String u = pq.poll();
            if (settled.contains(u)) continue;
            settled.add(u);
            System.out.println("Settle " + u + " (dist=" + dist.get(u) + ")");

            for (Edge e : graph.get(u)) {
                int alt = dist.get(u) + e.cost;
                if (alt < dist.get(e.to)) {
                    dist.put(e.to, alt);
                    prev.put(e.to, u);
                    pq.add(e.to);
                    System.out.println("  Update " + e.to + ": dist=" + alt + " via " + u);
                }
            }
        }

        System.out.println("\n=== Shortest Distances from " + start + " ===");
        for (String node : new TreeSet<>(dist.keySet())) {
            List<String> path = new ArrayList<>();
            String p = node;
            while (p != null) { path.add(p); p = prev.get(p); }
            Collections.reverse(path);
            System.out.println(node + ": dist=" + dist.get(node) + "  path=" + String.join("->", path));
        }
    }
}
Main.main(null);

In [ ]:
%%html

<!-- UI_RUNNER: Dijkstra's Algorithm Knowledge Check -->

<style>
.quiz-container { margin: 1.5rem 0; }
.question { margin-bottom: 1rem; padding: 1rem; background: #1a1a1a; border: 1px solid #333; border-radius: 6px; border-left: 4px solid #2a7ae4; }
.question-title { font-weight: 600; font-size: 0.95rem; color: #fff; margin-bottom: 0.75rem; }
.question label { display: block; padding: 0.3rem 0; color: #aaa; cursor: pointer; font-size: 0.85rem; }
.question label:hover { color: #fff; }
.quiz-btn { padding: 0.6rem 1.4rem; background: #2a7ae4; color: #fff; border: none; border-radius: 4px; cursor: pointer; font-size: 0.95rem; margin-top: 0.5rem; }
.quiz-result { margin-top: 1rem; padding: 0.75rem 1rem; border-radius: 4px; font-weight: 600; font-size: 0.9rem; }
.unlock-notice { display: none; margin-top: 1rem; padding: 0.75rem 1rem; background: #1a2a3a; border: 1px solid #2a7ae4; border-radius: 4px; color: #7ab8f5; font-size: 0.9rem; }
.unlock-notice.show { display: block; }
</style>

<div class="quiz-container" id="dijkstra-quiz-container">
  <h3 style="margin-bottom:0.75rem;font-size:1.05rem;">Knowledge Check: Dijkstra's Algorithm</h3>

  <div class="question">
    <div class="question-title">Q1. Dijkstra's always expands the node with the smallest...</div>
    <label><input type="radio" name="qd1" value="a"> a) h(n) &mdash; heuristic estimate to goal</label>
    <label><input type="radio" name="qd1" value="b"> b) g(n) &mdash; actual cost from start</label>
    <label><input type="radio" name="qd1" value="c"> c) f(n) = g(n) + h(n)</label>
    <label><input type="radio" name="qd1" value="d"> d) Number of neighbors</label>
  </div>

  <div class="question">
    <div class="question-title">Q2. Why is Dijkstra's slower than A* for single-target pathfinding?</div>
    <label><input type="radio" name="qd2" value="a"> a) It uses a worse data structure</label>
    <label><input type="radio" name="qd2" value="b"> b) It explores all directions equally with no heuristic to guide it toward the goal</label>
    <label><input type="radio" name="qd2" value="c"> c) It does not use a priority queue</label>
    <label><input type="radio" name="qd2" value="d"> d) It cannot handle weighted graphs</label>
  </div>

  <button class="quiz-btn" id="dijkstra-quiz-btn">Submit</button>
  <div id="dijkstra-quiz-result" class="quiz-result" style="display:none;"></div>
  <div id="dijkstra-quiz-unlock" class="unlock-notice">&#x2705; A* Algorithm unlocked! Click it in the sidebar to continue.</div>
</div>

<script>
(function() {
  var btn = document.getElementById('dijkstra-quiz-btn');
  if (!btn) return;
  btn.addEventListener('click', function() {
    var container = document.getElementById('dijkstra-quiz-container');
    var q1 = container.querySelector('input[name="qd1"]:checked');
    var q2 = container.querySelector('input[name="qd2"]:checked');
    var result = document.getElementById('dijkstra-quiz-result');
    var unlock = document.getElementById('dijkstra-quiz-unlock');

    if (!q1 || !q2) {
      result.style.display = 'block';
      result.style.background = '#3a1a1a';
      result.style.border = '1px solid #f44336';
      result.style.color = '#f44336';
      result.textContent = 'Please answer all questions.';
      return;
    }

    var score = 0;
    if (q1.value === 'b') score++;
    if (q2.value === 'b') score++;

    result.style.display = 'block';
    if (score === 2) {
      result.style.background = '#1a3a1a';
      result.style.border = '1px solid #4CAF50';
      result.style.color = '#4CAF50';
      result.textContent = 'Excellent! 2/2 — You understand Dijkstra's algorithm.';
      unlock.style.display = 'block';
      window.dispatchEvent(new CustomEvent('quiz-complete', {
        detail: { zone: 'dijkstra', score: 2, nextZone: "astar" }
      }));
    } else {
      result.style.background = '#3a1a1a';
      result.style.border = '1px solid #f44336';
      result.style.color = '#f44336';
      result.textContent = score + '/2 — Review the material above and try again.';
    }
  });
})();
</script>

{% comment %} Dijkstra Zone - End {% endcomment %}

</div><!-- closes #zone-section-dijkstra -->

{% comment %} A* Zone - Start {% endcomment %}

<div id="zone-section-astar" class="zone-section" style="display: none;" markdown="1">

In [ ]:
%%html

<!-- UI_RUNNER: A* Lesson -->

<div class="lesson">
  <h2>A* Algorithm</h2>
  <p>A* (pronounced "A-star") is the <strong>best of both worlds</strong> — it combines Dijkstra's guaranteed shortest path with Greedy's heuristic guidance. It is the most widely used pathfinding algorithm in practice.</p>

  <h3>Core formula</h3>
  <blockquote>
    <strong>f(n) = g(n) + h(n)</strong><br>
    &bull; <code>g(n)</code> — actual cost from start to node n (like Dijkstra)<br>
    &bull; <code>h(n)</code> — estimated cost from n to goal (like Greedy)<br>
    &bull; <code>f(n)</code> — total estimated cost of the cheapest path through n
  </blockquote>

  <h3>Algorithm steps</h3>
  <ol>
    <li>Set <code>g(start) = 0</code>, <code>f(start) = h(start)</code></li>
    <li>Add start to a min-priority queue sorted by <code>f(n)</code></li>
    <li>Pop the node with the smallest <code>f(n)</code></li>
    <li>If it is the goal — reconstruct and return the path</li>
    <li>For each neighbor: compute <code>tentative_g = g(current) + edge_cost</code></li>
    <li>If <code>tentative_g &lt; g(neighbor)</code>, update g, f, parent — re-add to queue</li>
    <li>Repeat from step 3</li>
  </ol>

  <h3>Key properties</h3>
  <table>
    <tr><th>Property</th><th>A*</th></tr>
    <tr><td><strong>Uses</strong></td><td><code>f(n) = g(n) + h(n)</code></td></tr>
    <tr><td><strong>Complete?</strong></td><td>Yes</td></tr>
    <tr><td><strong>Optimal?</strong></td><td>Yes — if <code>h(n)</code> is <strong>admissible</strong></td></tr>
    <tr><td><strong>Time</strong></td><td>O(b<sup>d</sup>) — much better than Dijkstra for single target</td></tr>
    <tr><td><strong>Heuristic</strong></td><td>Yes — steers search toward the goal</td></tr>
  </table>

  <h3>Admissibility — the key constraint</h3>
  <p>A heuristic is <strong>admissible</strong> if it never overestimates the true cost to reach the goal. If it overestimates, A* may skip the optimal path.</p>
  <ul>
    <li><strong>Manhattan distance</strong> — grid maps without diagonal moves: <code>|dx| + |dy|</code></li>
    <li><strong>Euclidean distance</strong> — straight-line distance in open space</li>
  </ul>

  <div class="callout">
    <strong>Connection to the Maze Runner:</strong> The maze on the right uses A* with Manhattan distance as <code>h(n)</code>. Watch it find the shortest path — it expands far fewer nodes than Dijkstra would because the heuristic keeps it focused on the exit.
  </div>
</div>

In [ ]:
// CODE_RUNNER: A* Algorithm

import java.util.*;

public class Main {

    static class Node implements Comparable<Node> {
        String name;
        double gScore;
        double fScore;
        Node parent;

        public Node(String name) {
            this.name = name;
            this.gScore = Double.MAX_VALUE;
            this.fScore = Double.MAX_VALUE;
            this.parent = null;
        }

        public int compareTo(Node other) {
            return Double.compare(this.fScore, other.fScore);
        }

        public int hashCode() { return name.hashCode(); }
        public boolean equals(Object obj) {
            if (obj instanceof Node) return name.equals(((Node) obj).name);
            return false;
        }
    }

    static class Edge {
        String to;
        double cost;
        Edge(String to, double cost) { this.to = to; this.cost = cost; }
    }

    public static List<String> findPath(Map<String, List<Edge>> graph,
                                        Map<String, Double> heuristic,
                                        String start, String goal) {
        PriorityQueue<Node> openSet = new PriorityQueue<>();
        Set<String> closedSet = new HashSet<>();
        Map<String, Node> allNodes = new HashMap<>();
        int nodesExplored = 0;

        Node startNode = new Node(start);
        startNode.gScore = 0;
        startNode.fScore = heuristic.getOrDefault(start, 0.0);
        openSet.add(startNode);
        allNodes.put(start, startNode);

        while (!openSet.isEmpty()) {
            Node current = openSet.poll();
            nodesExplored++;

            System.out.println("Exploring: " + current.name +
                " (g=" + (int)current.gScore + ", h=" + heuristic.getOrDefault(current.name, 0.0).intValue() +
                ", f=" + (int)current.fScore + ")");

            if (current.name.equals(goal)) {
                System.out.println("\nGoal reached! Nodes explored: " + nodesExplored);
                List<String> path = new ArrayList<>();
                Node c = current;
                while (c != null) { path.add(c.name); c = c.parent; }
                Collections.reverse(path);
                return path;
            }

            closedSet.add(current.name);

            for (Edge edge : graph.getOrDefault(current.name, new ArrayList<>())) {
                if (closedSet.contains(edge.to)) continue;
                Node neighbor = allNodes.getOrDefault(edge.to, new Node(edge.to));
                if (!allNodes.containsKey(edge.to)) allNodes.put(edge.to, neighbor);

                double tentativeG = current.gScore + edge.cost;
                if (tentativeG < neighbor.gScore) {
                    neighbor.parent = current;
                    neighbor.gScore = tentativeG;
                    neighbor.fScore = tentativeG + heuristic.getOrDefault(edge.to, 0.0);
                    if (!openSet.contains(neighbor)) openSet.add(neighbor);
                }
            }
        }
        return new ArrayList<>();
    }

    public static void main(String[] args) {
        Map<String, List<Edge>> graph = new HashMap<>();
        graph.put("A", Arrays.asList(new Edge("B", 4), new Edge("C", 2)));
        graph.put("B", Arrays.asList(new Edge("D", 3)));
        graph.put("C", Arrays.asList(new Edge("D", 6)));
        graph.put("D", new ArrayList<>());

        Map<String, Double> heuristic = new HashMap<>();
        heuristic.put("A", 5.0);
        heuristic.put("B", 2.0);
        heuristic.put("C", 4.0);
        heuristic.put("D", 0.0);

        System.out.println("=== A* Algorithm ===");
        System.out.println("f(n) = g(n) + h(n)\n");
        List<String> path = findPath(graph, heuristic, "A", "D");
        System.out.println("\nOptimal path: " + String.join(" -> ", path));
        System.out.println("Total cost: 7  (A->B: 4, B->D: 3)");
        System.out.println("\nNote: A* skipped A->C->D (cost=8) guided by the heuristic.");
    }
}
Main.main(null);

In [ ]:
%%html

<!-- UI_RUNNER: A* Algorithm Knowledge Check -->

<style>
.quiz-container { margin: 1.5rem 0; }
.question { margin-bottom: 1rem; padding: 1rem; background: #1a1a1a; border: 1px solid #333; border-radius: 6px; border-left: 4px solid #2a7ae4; }
.question-title { font-weight: 600; font-size: 0.95rem; color: #fff; margin-bottom: 0.75rem; }
.question label { display: block; padding: 0.3rem 0; color: #aaa; cursor: pointer; font-size: 0.85rem; }
.question label:hover { color: #fff; }
.quiz-btn { padding: 0.6rem 1.4rem; background: #2a7ae4; color: #fff; border: none; border-radius: 4px; cursor: pointer; font-size: 0.95rem; margin-top: 0.5rem; }
.quiz-result { margin-top: 1rem; padding: 0.75rem 1rem; border-radius: 4px; font-weight: 600; font-size: 0.9rem; }
.unlock-notice { display: none; margin-top: 1rem; padding: 0.75rem 1rem; background: #1a2a3a; border: 1px solid #2a7ae4; border-radius: 4px; color: #7ab8f5; font-size: 0.9rem; }
.unlock-notice.show { display: block; }
</style>

<div class="quiz-container" id="astar-quiz-container">
  <h3 style="margin-bottom:0.75rem;font-size:1.05rem;">Knowledge Check: A* Algorithm</h3>

  <div class="question">
    <div class="question-title">Q1. A* uses f(n) = g(n) + h(n). What does g(n) represent?</div>
    <label><input type="radio" name="qa1" value="a"> a) Estimated cost from n to goal</label>
    <label><input type="radio" name="qa1" value="b"> b) Actual cost from start to n</label>
    <label><input type="radio" name="qa1" value="c"> c) Total number of neighbors of n</label>
    <label><input type="radio" name="qa1" value="d"> d) The heuristic weight</label>
  </div>

  <div class="question">
    <div class="question-title">Q2. Which condition must h(n) satisfy for A* to guarantee an optimal path?</div>
    <label><input type="radio" name="qa2" value="a"> a) h(n) must always equal the true cost</label>
    <label><input type="radio" name="qa2" value="b"> b) h(n) must be consistent (monotone)</label>
    <label><input type="radio" name="qa2" value="c"> c) h(n) must be admissible &mdash; never overestimate the true cost</label>
    <label><input type="radio" name="qa2" value="d"> d) h(n) must always be zero</label>
  </div>

  <button class="quiz-btn" id="astar-quiz-btn">Submit</button>
  <div id="astar-quiz-result" class="quiz-result" style="display:none;"></div>
  <div id="astar-quiz-unlock" class="unlock-notice">&#x1F389; You have completed all three algorithms! Try the Maze Runner on the right.</div>
</div>

<script>
(function() {
  var btn = document.getElementById('astar-quiz-btn');
  if (!btn) return;
  btn.addEventListener('click', function() {
    var container = document.getElementById('astar-quiz-container');
    var q1 = container.querySelector('input[name="qa1"]:checked');
    var q2 = container.querySelector('input[name="qa2"]:checked');
    var result = document.getElementById('astar-quiz-result');
    var unlock = document.getElementById('astar-quiz-unlock');

    if (!q1 || !q2) {
      result.style.display = 'block';
      result.style.background = '#3a1a1a';
      result.style.border = '1px solid #f44336';
      result.style.color = '#f44336';
      result.textContent = 'Please answer all questions.';
      return;
    }

    var score = 0;
    if (q1.value === 'b') score++;
    if (q2.value === 'c') score++;

    result.style.display = 'block';
    if (score === 2) {
      result.style.background = '#1a3a1a';
      result.style.border = '1px solid #4CAF50';
      result.style.color = '#4CAF50';
      result.textContent = 'Perfect! 2/2 — You have mastered A*!';
      unlock.style.display = 'block';
      window.dispatchEvent(new CustomEvent('quiz-complete', {
        detail: { zone: 'astar', score: 2, nextZone: null }
      }));
    } else {
      result.style.background = '#3a1a1a';
      result.style.border = '1px solid #f44336';
      result.style.color = '#f44336';
      result.textContent = score + '/2 — Review the material above and try again.';
    }
  });
})();
</script>

{% comment %} A* Zone - End {% endcomment %}

</div><!-- closes #zone-section-astar -->